# Search Indicators By Tags

This notebook filters `htoc_observed_indicator_tags.csv` by a tag search string and returns the matching indicator IDs (and optionally the matching rows).

## Configure parameters below
Set `tag_search` and run the notebook.

In [31]:
# -------------------------
# Parameters (edit + run)
# -------------------------

tags_file_path = r"Z:\\HTOC\\Data_Analytics\\Data\\Observed_Tags\\htoc_observed_indicator_tags.csv"

# Which column to search for your tag string:
# - "tag" is the cleaned/normalized tag value
# - "orig_tag" is the original raw tag value
tag_column = "tag"  # "tag" | "orig_tag"

# Your tag search term (e.g. "phishing", "Malspam", "Ransomware")
# Leave empty to be prompted at runtime.
tag_search = ""
if not str(tag_search).strip():
    tag_search = input("Enter tag search criteria (comma-separated for multiple): ").strip()

# - "contains" performs a literal substring match
# - "exact" requires full-value equality
match_mode = "contains"  # "contains" | "exact"

# When multiple tag terms are provided (comma-separated), choose how to combine them:
# - "any"  => row matches if it matches ANY term (OR)
# - "all"  => row matches only if it matches ALL terms (AND)
multi_input_mode = "any"  # "any" | "all"

# Case-insensitive by default (matches the script we added)
case_sensitive = False

# Chunk size for streaming the CSV (avoids loading everything at once)
chunksize = 5000

# Outputs (optional)
output_indicators_csv = None  # e.g. r"Z:\\HTOC\\...\\indicators.csv"
output_matching_rows_csv = None  # e.g. r"Z:\\HTOC\\...\\matching_rows.csv"

# Display settings
show_indicators_preview = 50

# Matching-row display settings (filtered rows can be large)
show_matching_rows_preview = 50
show_full_matching_rows_if_under = 200

# Threat score lookup (by indicator)
threat_scores_excel_path = r"Z:\\HTOC\\Data_Analytics\\Data\\Threat Assessment Scores\\Threat_Assessment_Scores.xlsx"

# When True, the notebook loads `threat_scores_excel_path`, filters rows whose
# indicator matches the tag-search `indicators`, and outputs those rows.
show_scores_records = True

# Optional: write the matching score rows
output_scores_filtered_csv = None  # e.g. r"Z:\\HTOC\\...\\Threat_Assessment_Scores_filtered.csv"

# Optional: show the raw matching tag rows from htoc_observed_indicator_tags.csv
show_tag_matching_rows = False

# Optional: show the unique indicator list
show_indicators_list = False

In [32]:
from __future__ import annotations

import pandas as pd

# Helper: filter rows where the chosen tag column matches the search term.

def _normalize_tag_series(s: pd.Series) -> pd.Series:
    # Keep missing values as <NA> while ensuring .str operations work.
    return s.astype("string").str.strip()


def filter_by_tag(
    df: pd.DataFrame,
    search: str | list[str],
    *,
    tag_column: str,
    match_mode: str,
    case_sensitive: bool,
    multi_input_mode: str,
) -> pd.DataFrame:
    if tag_column not in df.columns:
        raise KeyError(
            f"tag_column '{tag_column}' not found. Available columns: {list(df.columns)}"
        )

    # Normalize search input into a list of non-empty terms.
    if isinstance(search, (list, tuple, set)):
        search_terms = [str(x).strip() for x in search if str(x).strip()]
    else:
        search_terms = [str(search).strip()]

    if not search_terms:
        raise ValueError("tag search must be non-empty")

    if multi_input_mode not in {"any", "all"}:
        raise ValueError("multi_input_mode must be 'any' or 'all'")

    tags = _normalize_tag_series(df[tag_column])

    def _mask_for_term(term: str) -> pd.Series:
        if match_mode == "exact":
            if case_sensitive:
                return tags == term
            return tags.str.lower() == term.lower()

        if match_mode == "contains":
            # regex=False so the search term is treated literally
            return tags.str.contains(term, case=case_sensitive, na=False, regex=False)

        raise ValueError(f"Unknown match_mode: {match_mode}")

    mask = None
    for term in search_terms:
        term_mask = _mask_for_term(term)
        if mask is None:
            mask = term_mask
        elif multi_input_mode == "any":
            mask = mask | term_mask
        else:  # multi_input_mode == "all"
            mask = mask & term_mask

    # At this point, mask is always set because search_terms is non-empty.
    return df.loc[mask].copy()


In [33]:
import os

# Best-effort display helper (works in notebooks, harmless in scripts)
try:
    from IPython.display import display, clear_output  # type: ignore
except Exception:
    display = None  # type: ignore
    clear_output = None  # type: ignore

# Load the observed indicator tags CSV
print(f"Loading: {tags_file_path}")
if not os.path.exists(tags_file_path):
    raise FileNotFoundError(f"CSV not found: {tags_file_path}")

# Stream-filter the CSV so we don't load everything into memory at once
print("Scanning CSV in chunks...")

match_chunks = []
indicator_seen = set()
indicators: list[str] = []

# Parse tag search terms (comma/newline separated)
tag_terms = [
    t
    for t in str(tag_search).replace("\n", ",").split(",")
    if str(t).strip()
]

if not tag_terms:
    raise ValueError("No tag search terms provided")

print(f"Searching for terms: {tag_terms} (mode={multi_input_mode}, match_mode={match_mode})")

# Grab columns once (cheap; reads no rows)
columns = pd.read_csv(tags_file_path, dtype=str, nrows=0).columns

for chunk_idx, chunk in enumerate(
    pd.read_csv(tags_file_path, dtype=str, chunksize=chunksize), start=1
):
    filtered_chunk = filter_by_tag(
        chunk,
        tag_terms,
        tag_column=tag_column,
        match_mode=match_mode,
        case_sensitive=case_sensitive,
        multi_input_mode=multi_input_mode,
    )

    if not filtered_chunk.empty:
        match_chunks.append(filtered_chunk)

        # Pull indicator ids, preserving first-seen order
        if "indicator" in filtered_chunk.columns:
            for ind in (
                filtered_chunk["indicator"].astype("string").dropna().str.strip()
            ):
                if ind != "" and ind not in indicator_seen:
                    indicator_seen.add(ind)
                    indicators.append(ind)

    # Light progress (keeps notebook output readable)
    if chunk_idx % 20 == 0:
        print(f"Processed {chunk_idx} chunks...")

filtered = (
    pd.concat(match_chunks, ignore_index=True)
    if match_chunks
    else pd.DataFrame(columns=columns)
)

# Sort matching rows by the selected tag column
if not filtered.empty and tag_column in filtered.columns:
    filtered = filtered.copy()

    # Normalize sort keys to avoid issues with trailing/leading whitespace.
    filtered["_sort_tag"] = filtered[tag_column].astype("string").str.strip()

    sort_cols = ["_sort_tag"]
    if "indicator" in filtered.columns:
        filtered["_sort_indicator"] = (
            filtered["indicator"].astype("string").str.strip()
        )
        sort_cols.append("_sort_indicator")

    filtered = (
        filtered.sort_values(
            by=sort_cols,
            kind="mergesort",  # stable sort
            na_position="last",
        )
        .drop(
            columns=[c for c in ["_sort_tag", "_sort_indicator"] if c in filtered.columns],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    # Rebuild indicator list in sorted order (unique)
    if "indicator" in filtered.columns:
        _ind_series = filtered["indicator"].astype("string").str.strip()
        _ind_series = _ind_series.loc[_ind_series.notna() & (_ind_series != "")]
        indicators = _ind_series.drop_duplicates().tolist()

print(
    "Matches: "
    f"{len(filtered)} rows, {len(indicators)} unique indicators "
    f"(tag_column={tag_column}, match_mode={match_mode})."
)

# Optional outputs: indicator list
indicators_df = pd.DataFrame({"indicator": indicators})
if show_indicators_list:
    if display is not None:
        display(indicators_df)
    else:
        print(indicators_df)

# Optional: load and filter the Threat Assessment Scores Excel by indicator
scores_df_filtered = None
if show_scores_records or output_scores_filtered_csv:
    print(f"Loading scores Excel: {threat_scores_excel_path}")
    if not os.path.exists(threat_scores_excel_path):
        raise FileNotFoundError(
            f"Scores Excel not found: {threat_scores_excel_path}"
        )

    scores_df = pd.read_excel(threat_scores_excel_path)

    # Match indicator column name robustly
    _indicator_col = next(
        (c for c in ["indicator", "Indicator", "INDICATOR"] if c in scores_df.columns),
        None,
    )
    if _indicator_col is None:
        raise KeyError(
            "Could not find an indicator column in the scores Excel. "
            f"Columns: {list(scores_df.columns)}"
        )

    indicator_set = set(indicators)

    # Normalize for reliable comparison
    scores_df["_indicator_norm"] = (
        scores_df[_indicator_col].astype("string").str.strip()
    )

    scores_df_filtered = scores_df[scores_df["_indicator_norm"].isin(indicator_set)].copy()

    # Sort to match the tag-sorted indicator order.
    # Use an order mapping instead of DataFrame merge to avoid dtype mismatches
    # (e.g., indicator-like strings inferred as floats by pandas).
    order_map = {ind: i for i, ind in enumerate(indicators)}
    scores_df_filtered["_order"] = scores_df_filtered["_indicator_norm"].map(order_map)

    scores_df_filtered = (
        scores_df_filtered.sort_values(
            "_order", kind="mergesort", na_position="last"
        )
        .drop(columns=["_indicator_norm", "_order"])
        .reset_index(drop=True)
    )

    print(
        "Scores rows matched: "
        f"{len(scores_df_filtered)} (indicator_column={_indicator_col})."
    )

    if show_scores_records:
        if display is not None:
            display(scores_df_filtered)
        else:
            print(scores_df_filtered)

    if output_scores_filtered_csv:
        scores_df_filtered.to_csv(output_scores_filtered_csv, index=False)
        print(f"Wrote scores filtered CSV: {output_scores_filtered_csv}")

# Optional: show raw matching tag rows
if show_tag_matching_rows:
    if display is not None:
        display(filtered)
    else:
        print(filtered)

# Optional file outputs for tags/indicators
if output_matching_rows_csv:
    pd.DataFrame(filtered).to_csv(output_matching_rows_csv, index=False)
    print(f"Wrote matching rows: {output_matching_rows_csv}")

if output_indicators_csv:
    pd.DataFrame({"indicator": indicators}).to_csv(output_indicators_csv, index=False)
    print(f"Wrote unique indicators: {output_indicators_csv}")

# Convenience: show number of indicators
print(f"Total unique indicators: {len(indicators)}")

# Interactive loop: allow multiple searches in one notebook run.
# (Your first search already executed above; this handles subsequent searches.)

_scores_df = None
_indicator_col = None
if show_scores_records or output_scores_filtered_csv:
    # Load scores once for subsequent searches.
    if not os.path.exists(threat_scores_excel_path):
        raise FileNotFoundError(
            f"Scores Excel not found: {threat_scores_excel_path}"
        )

    _scores_df = pd.read_excel(threat_scores_excel_path)

    _indicator_col = next(
        (c for c in ["indicator", "Indicator", "INDICATOR"] if c in _scores_df.columns),
        None,
    )
    if _indicator_col is None:
        raise KeyError(
            "Could not find an indicator column in the scores Excel. "
            f"Columns: {list(_scores_df.columns)}"
        )

    _scores_df["_indicator_norm"] = (
        _scores_df[_indicator_col].astype("string").str.strip()
    )


def _run_search_once(current_tag_search: str) -> None:
    match_chunks = []
    indicator_seen = set()
    indicators: list[str] = []

    tag_terms = [
        t
        for t in str(current_tag_search).replace("\n", ",").split(",")
        if str(t).strip()
    ]
    if not tag_terms:
        raise ValueError("No tag search terms provided")

    print(
        f"\nSearching for terms: {tag_terms} (mode={multi_input_mode}, match_mode={match_mode})."
    )

    for chunk_idx, chunk in enumerate(
        pd.read_csv(tags_file_path, dtype=str, chunksize=chunksize), start=1
    ):
        filtered_chunk = filter_by_tag(
            chunk,
            tag_terms,
            tag_column=tag_column,
            match_mode=match_mode,
            case_sensitive=case_sensitive,
            multi_input_mode=multi_input_mode,
        )

        if not filtered_chunk.empty:
            match_chunks.append(filtered_chunk)

            if "indicator" in filtered_chunk.columns:
                for ind in (
                    filtered_chunk["indicator"].astype("string").dropna().str.strip()
                ):
                    if ind != "" and ind not in indicator_seen:
                        indicator_seen.add(ind)
                        indicators.append(ind)

        if chunk_idx % 20 == 0:
            print(f"Processed {chunk_idx} chunks...")

    filtered = (
        pd.concat(match_chunks, ignore_index=True)
        if match_chunks
        else pd.DataFrame(columns=columns)
    )

    # Sort matching rows by the selected tag column
    if not filtered.empty and tag_column in filtered.columns:
        filtered = filtered.copy()
        filtered["_sort_tag"] = filtered[tag_column].astype("string").str.strip()

        sort_cols = ["_sort_tag"]
        if "indicator" in filtered.columns:
            filtered["_sort_indicator"] = (
                filtered["indicator"].astype("string").str.strip()
            )
            sort_cols.append("_sort_indicator")

        filtered = (
            filtered.sort_values(
                by=sort_cols,
                kind="mergesort",
                na_position="last",
            )
            .drop(
                columns=[
                    c
                    for c in ["_sort_tag", "_sort_indicator"]
                    if c in filtered.columns
                ],
                errors="ignore",
            )
            .reset_index(drop=True)
        )

        if "indicator" in filtered.columns:
            _ind_series = filtered["indicator"].astype("string").str.strip()
            _ind_series = _ind_series.loc[
                _ind_series.notna() & (_ind_series != "")
            ]
            indicators = _ind_series.drop_duplicates().tolist()

    print(
        "Matches: "
        f"{len(filtered)} rows, {len(indicators)} unique indicators "
        f"(tag_column={tag_column}, match_mode={match_mode})."
    )

    if show_indicators_list:
        indicators_df = pd.DataFrame({"indicator": indicators})
        if display is not None:
            display(indicators_df)
        else:
            print(indicators_df)

    # Scores lookup (optional)
    if _scores_df is not None:
        indicator_set = set(indicators)
        scores_df_filtered = _scores_df[
            _scores_df["_indicator_norm"].isin(indicator_set)
        ].copy()

        order_map = {ind: i for i, ind in enumerate(indicators)}
        scores_df_filtered["_order"] = scores_df_filtered["_indicator_norm"].map(order_map)

        scores_df_filtered = (
            scores_df_filtered.sort_values(
                "_order", kind="mergesort", na_position="last"
            )
            .drop(columns=["_indicator_norm", "_order"])
            .reset_index(drop=True)
        )

        print(
            "Scores rows matched: "
            f"{len(scores_df_filtered)} (indicator_column={_indicator_col})."
        )

        if show_scores_records:
            if display is not None:
                display(scores_df_filtered)
            else:
                print(scores_df_filtered)

        if output_scores_filtered_csv:
            scores_df_filtered.to_csv(output_scores_filtered_csv, index=False)
            print(f"Wrote scores filtered CSV: {output_scores_filtered_csv}")

    if show_tag_matching_rows:
        if display is not None:
            display(filtered)
        else:
            print(filtered)

    if output_matching_rows_csv:
        pd.DataFrame(filtered).to_csv(output_matching_rows_csv, index=False)
        print(f"Wrote matching rows: {output_matching_rows_csv}")

    if output_indicators_csv:
        pd.DataFrame({"indicator": indicators}).to_csv(output_indicators_csv, index=False)
        print(f"Wrote unique indicators: {output_indicators_csv}")

    print(f"Total unique indicators: {len(indicators)}")


while True:
    # Press Enter on this prompt to exit the search loop.
    next_search = input(
        "Enter tag search criteria (comma-separated). Press Enter to end: "
    ).strip()

    if not next_search:
        break

    # Replace previous outputs for a single-run UX.
    if clear_output is not None:
        clear_output(wait=True)

    _run_search_once(next_search)



Searching for terms: ['malspam', ' iran'] (mode=any, match_mode=contains).
Matches: 666 rows, 666 unique indicators (tag_column=tag, match_mode=contains).
Scores rows matched: 269 (indicator_column=Indicator).


,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation
0,1.4.195.14,2026-01-17,Address,12,1,0.999342,1,0,"CMS, VA",NaN,NaN,180,229,134,low,Severity: low. VT score: 1. Top drivers: TOR a...
1,1.54.101.176,2025-12-04,Address,1,1,0.999945,1,0,NaN,NaN,NaN,170,196,111,low,Severity: low. VT score: 0. Top drivers: TOR a...
2,102.0.5.152,2025-12-16,Address,17,1,0.999068,1,0,"CMS, DHA, VA",NaN,NaN,170,224,141,low,Severity: low. VT score: 2. Top drivers: TOR a...
3,102.164.252.150,2025-10-29,Address,4,1,0.999781,1,0,"CMS, NIH, VA",NaN,NaN,170,224,134,low,Severity: low. VT score: 1. Top drivers: TOR a...
4,102.209.18.96,2025-12-15,Address,11,1,0.999397,1,0,"FDA, VA",NaN,NaN,170,224,134,low,Severity: low. VT score: 1. Top drivers: TOR a...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,miledin.mwhelp.site,2026-02-18,Host,4,3,0.999781,0,0,NIH,NaN,UNC5952,180,416,90,low,[2026-03-16] Severity: low. VT score not avail...
265,pilwerui.rchelp.top,2026-02-13,Host,18,3,0.999014,0,0,DHA,NaN,UNC5952,180,590,91,low,[2026-03-11] Severity: low. VT score not avail...
266,sicil1.org,2026-03-04,Host,16,3,0.999123,0,0,"NIH, VA",NaN,NaN,170,464,30,low,[2026-03-16] Severity: low. VT score not avail...
267,troy.humphrey@yourlscgroup.com,2026-02-20,EmailAddress,1,3,0.999945,0,0,VA,NaN,NaN,170,557,21,low,[2026-03-16] Severity: low. VT score not avail...


Total unique indicators: 666
